# Statistical analysis of the composition results


In [71]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd().parents[1]

In [72]:
# Load data
with open(REPO_ROOT / "results/composition_scoring_l17_summary.json") as f:
    summary = json.load(f)
  
# Metadata  
print(summary["model"], summary["layer"], summary["alpha"], summary["tau_value"])

# Per-composition entries
pairs = summary["pairs"]
print(pairs[0].keys)

# Extract per-composition data
rows = []
for p in pairs:
    if p.get("status") != "ok":
        continue
    rows.append({
        "trait_a": p["trait_a"], "trait_b": p["trait_b"],
        "cos": p["cos"], "regime": p["regime"],
        "comp_base": p["baseline"]["composition_mean"],
        "comp_steered": p["steered"]["composition_mean"],
        "coh_steered": p["steered"]["coherence_mean"],
        "delta_a_joint":  p["delta"]["trait_a_joint"],
        "delta_a_single": p["delta"]["trait_a_single"],
        "delta_b_joint":  p["delta"]["trait_b_joint"],
        "delta_b_single": p["delta"]["trait_b_single"],
        "delta_comp": p["delta"]["composition"],
        "delta_coh":  p["delta"]["coherence"],
    })
    
# Convert to pd dataframe
df = pd.DataFrame(rows)
df["regime"].value_counts()

meta-llama/Llama-3.1-8B-Instruct 17 4.0 0.8900726269483555
<built-in method keys of dict object at 0x12f3f9080>


regime
mixed          19
emergent        6
dominant        5
additive        3
suppressive     3
Name: count, dtype: int64

In [73]:
# Add semantic similarity
with open(REPO_ROOT / "results/semantic_similarity.json") as f:
    sem = json.load(f)
    
sem_lookup = {
    tuple(sorted([p["trait_a"], p["trait_b"]])): p["sem_sim"]
    for p in sem["pairs"]
}

df["sem_sim"] = df.apply(
    lambda r: sem_lookup[tuple(sorted([r["trait_a"], r["trait_b"]]))],
    axis=1,
)

In [74]:
# Look at dataframe
df.head(n=40)

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim
0,apathetic,confidence,0.0143,dominant,32.70,54.85,55.56,57.13,32.65,-12.84,20.08,22.15,-42.23,0.240104
1,apathetic,evil,0.2578,mixed,4.79,54.88,28.75,73.73,36.76,26.44,45.90,50.09,-68.62,0.321475
2,apathetic,formality,0.0150,mixed,44.56,78.29,80.87,64.24,21.48,3.22,7.85,33.73,-16.61,0.246348
3,apathetic,hallucinating,-0.1642,emergent,10.37,56.44,59.95,43.39,23.64,48.76,28.16,46.07,-32.51,0.198113
4,apathetic,humorous,0.1923,mixed,1.25,65.28,27.09,78.35,22.34,49.71,75.80,64.03,-69.25,0.232589
5,apathetic,impolite,0.6948,mixed,0.93,76.79,23.47,80.17,28.00,71.56,66.36,75.86,-72.93,0.469881
6,apathetic,power_seeking,0.0063,additive,10.27,25.73,81.73,40.38,34.56,-9.46,-8.23,15.46,-15.53,0.291267
7,apathetic,sycophantic,0.0503,mixed,3.96,51.37,43.79,62.99,34.05,31.84,69.07,47.42,-53.52,0.344080
8,confidence,evil,0.2294,dominant,28.39,53.26,34.45,-14.58,24.39,64.31,37.09,24.87,-62.72,0.209184
9,confidence,formality,0.2262,mixed,75.96,82.50,78.46,10.44,17.56,2.64,5.60,6.54,-18.95,0.398346


### Cross-tab for cosine v. regime

We perform a descriptive analysis to see whether simple cross-tabulation predicts a regime using cosine similarity (Gram) matrix $G$.

In [75]:
# Add column with absolute value of cosine similarities
df['cosine_abs'] = df['cos'].abs()

# Bin cosine with the same strata of the previous analysis
df['cos_bin'] = pd.cut(
    df['cosine_abs'],
    bins=[0, 0.15, 0.30, 0.40],
    labels=['low', 'moderate', 'high']
)

df.head()

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim,cosine_abs,cos_bin
0,apathetic,confidence,0.0143,dominant,32.70,54.85,55.56,57.13,32.65,-12.84,20.08,22.15,-42.23,0.240104,0.0143,low
1,apathetic,evil,0.2578,mixed,4.79,54.88,28.75,73.73,36.76,26.44,45.90,50.09,-68.62,0.321475,0.2578,moderate
2,apathetic,formality,0.0150,mixed,44.56,78.29,80.87,64.24,21.48,3.22,7.85,33.73,-16.61,0.246348,0.0150,low
3,apathetic,hallucinating,-0.1642,emergent,10.37,56.44,59.95,43.39,23.64,48.76,28.16,46.07,-32.51,0.198113,0.1642,moderate
4,apathetic,humorous,0.1923,mixed,1.25,65.28,27.09,78.35,22.34,49.71,75.80,64.03,-69.25,0.232589,0.1923,moderate


In [76]:
# Cross-tab
pd.crosstab(df['cos_bin'], df['regime'])

regime,additive,dominant,emergent,mixed,suppressive
cos_bin,,,,,
low,2,3,2,4,1
moderate,1,2,3,7,1
high,0,0,1,4,0


In [77]:
# Statistical chi-square test on cross-tab
from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(
    pd.crosstab(df['cos_bin'], df['regime'])
)
print(f"\n chi2={chi2}, \n p={p}, \n dof={dof}, \n expected={expected}")


 chi2=4.681984126984126, 
 p=0.790962147758917, 
 dof=8, 
 expected=[[1.16129032 1.93548387 2.32258065 5.80645161 0.77419355]
 [1.35483871 2.25806452 2.70967742 6.77419355 0.90322581]
 [0.48387097 0.80645161 0.96774194 2.41935484 0.32258065]]


### Correlation between cosine and semantic similarities

In [85]:
corr = df[["cosine_abs", "sem_sim"]].corr().iloc[0, 1]
print(f"Pearson r(cosine_abs, sem_sim) = {corr:+.3f}")
print(df[["cosine_abs", "sem_sim"]].describe().round(3))

Pearson r(cosine_abs, sem_sim) = +0.528
       cosine_abs  sem_sim
count      36.000   36.000
mean        0.232    0.274
std         0.159    0.073
min         0.006    0.185
25%         0.105    0.222
50%         0.228    0.245
75%         0.316    0.322
max         0.695    0.470


## Logistic regression

We perform multiple logistic regressions to see whether the cosine similarity $|G_{i,j}|$ for composition of behavior $i$ and behavior $j$ is a predictor for the composition regime.
1. First, we perform a binary regression using only absolute cosine similarities, to predict additive v. non-additive;
2. We add semantic similarity as a confound control;
3. We repeat step 2 using signed cosine to see whether sign matters;
4. We try a multinomial logistic regression to predict all regimes.

In [78]:
# Save experiments results
results = []

In [79]:
# Add binary variable for additive v. non-additive
df['is_additive'] = (df['regime'] == 'additive').astype(int)
df.head()

,trait_a,trait_b,cos,regime,comp_base,comp_steered,coh_steered,delta_a_joint,delta_a_single,delta_b_joint,delta_b_single,delta_comp,delta_coh,sem_sim,cosine_abs,cos_bin,is_additive
0,apathetic,confidence,0.0143,dominant,32.70,54.85,55.56,57.13,32.65,-12.84,20.08,22.15,-42.23,0.240104,0.0143,low,0
1,apathetic,evil,0.2578,mixed,4.79,54.88,28.75,73.73,36.76,26.44,45.90,50.09,-68.62,0.321475,0.2578,moderate,0
2,apathetic,formality,0.0150,mixed,44.56,78.29,80.87,64.24,21.48,3.22,7.85,33.73,-16.61,0.246348,0.0150,low,0
3,apathetic,hallucinating,-0.1642,emergent,10.37,56.44,59.95,43.39,23.64,48.76,28.16,46.07,-32.51,0.198113,0.1642,moderate,0
4,apathetic,humorous,0.1923,mixed,1.25,65.28,27.09,78.35,22.34,49.71,75.80,64.03,-69.25,0.232589,0.1923,moderate,0


In [80]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold, LeaveOneOut

# Single rng instantiated
rng = np.random.default_rng(0)
N_BOOT  = 2000
N_PERM  = 10000

In [ ]:
def bootstrap_perm(model, X, y, exp_auc):

    def fit_auc(X_tr, y_tr, X_ev, y_ev):
        m = clone(model).fit(X_tr, y_tr)
        return roc_auc_score(y_ev, m.predict_proba(X_ev)[:, 1])

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_obs   = len(y)

    # --- LOO slope stability ---
    # For each of the 36 leave-one-out folds: fit on n-1 points, extract the
    # slope coefficient (coef_[0, :]), track sign and range.
    # Also count folds where the training set has < 2 positives — with 3
    # positives total and LOO, this only fires when a positive is held out AND
    # there are only 2 remaining. Expected count: 0 (we have 3 positives so
    # the worst case is 2 in training), but the check is a useful sanity gate
    # for future experiments with fewer positives.
    loo_slopes        = []
    loo_degen_folds   = 0
    for i in range(n_obs):
        mask   = np.ones(n_obs, dtype=bool)
        mask[i] = False
        y_tr   = y[mask]
        if (y_tr == 1).sum() < 2:
            loo_degen_folds += 1
            continue
        try:
            m = clone(model).fit(X[mask], y_tr)
            loo_slopes.append(m.coef_[0].tolist())    # one value per feature
        except Exception:
            pass

    loo_slopes_arr = np.array(loo_slopes)              # shape: (n_valid_folds, n_features)
    print(f"LOO folds with < 2 positives in training: {loo_degen_folds}")
    for feat_idx in range(loo_slopes_arr.shape[1]):
        col = loo_slopes_arr[:, feat_idx]
        print(f"  feature {feat_idx}: slope min={col.min():+.3f}  "
              f"max={col.max():+.3f}  "
              f"n_negative={int((col < 0).sum())}/{len(col)}")

    # --- Bootstrap 95% CI (stratified, evaluate on resample) ---
    boot_aucs = []
    for _ in range(N_BOOT):
        pi  = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        ni  = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([pi, ni])
        if len(np.unique(y[idx])) < 2:
            continue
        try:
            boot_aucs.append(fit_auc(X[idx], y[idx], X[idx], y[idx]))
        except ValueError:
            pass

    boot_aucs = np.array(boot_aucs)
    ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
    print(f"AUC = {exp_auc:.3f}  95% CI [{ci_lo:.3f}, {ci_hi:.3f}]  "
          f"(n_boot={len(boot_aucs)}  shape: min={boot_aucs.min():.3f} "
          f"median={np.median(boot_aucs):.3f} max={boot_aucs.max():.3f})")

    # --- Permutation p-value (one-sided) ---
    perm_aucs = []
    for _ in range(N_PERM):
        y_perm = rng.permutation(y)
        perm_aucs.append(fit_auc(X, y_perm, X, y_perm))

    perm_aucs = np.array(perm_aucs)
    p_value   = (np.sum(perm_aucs >= exp_auc) + 1) / (N_PERM + 1)
    print(f"permutation p (one-sided) = {p_value:.4f}")

    return {
        "auc_insample":           exp_auc,
        "ci_lo":                  float(ci_lo),
        "ci_hi":                  float(ci_hi),
        "n_boot":                 len(boot_aucs),
        "boot_aucs":              boot_aucs.tolist(),      # full distribution for figures
        "perm_p":                 float(p_value),
        "perm_auc_mean":          float(perm_aucs.mean()),
        "loo_slope_min":          float(loo_slopes_arr[:, 0].min()),
        "loo_slope_max":          float(loo_slopes_arr[:, 0].max()),
        "loo_slope_n_neg":        int((loo_slopes_arr[:, 0] < 0).sum()),
        "loo_slope_n_folds":      len(loo_slopes),
        "loo_degen_folds":        loo_degen_folds,
    }

In [82]:
scaler = StandardScaler()
binary_model = LogisticRegression(penalty=None)
multi_model = LogisticRegression(penalty=None, multi_class='multinomial')

### Experiment 1

In [83]:
# Load exp 1 data
X_abs   = df[['cosine_abs']].to_numpy()
X_abs_s = scaler.fit_transform(X_abs)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp1_results = binary_model.fit(X_abs_s, y_add)
y_proba_in   = exp1_results.predict_proba(X_abs_s)[:, 1]
exp1_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_abs_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp1_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp1_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp1_auc_loo:.3f}")
print(f"slope on cosine_abs (scaled) = {exp1_results.coef_[0,0]:+.3f}  "
      f"(negative => high |cos| reduces P(additive) — the geometric prediction)")

in-sample AUC      = 0.828
LOO (pooled) AUC   = 0.606
slope on cosine_abs (scaled) = -1.865  (negative => high |cos| reduces P(additive) — the geometric prediction)


In [84]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_abs_s, y_add, exp1_auc)
results.append({
    "experiment":  "1_abs_cos",
    "outcome":     "is_additive",
    "predictors":  "cosine_abs",
    "n_features":  1,
    "auc_loo":     exp1_auc_loo,
    "slope_abs_cos": float(exp1_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

AUC = 0.828  95% CI [0.606, 1.000]  (n_boot=2000)
permutation p (one-sided) = 0.0648


### Experiment 2

In [86]:
# Load exp 2 data
X_sem   = df[['sem_sim']].to_numpy()
X_sem_s = scaler.fit_transform(X_sem)
y_add   = df["is_additive"].to_numpy()

# Fit on all data for in-sample AUC + coefficient sign
exp2_results = binary_model.fit(X_sem_s, y_add)
y_proba_in   = exp2_results.predict_proba(X_sem_s)[:, 1]
exp2_auc     = roc_auc_score(y_add, y_proba_in)

# Leave-one-out: fit on n-1, predict held-out probability, pool 36 probabilities, single AUC
loo            = LeaveOneOut()
y_proba_loo    = cross_val_predict(binary_model, X_sem_s, y_add, cv=loo, method="predict_proba")[:, 1]
exp2_auc_loo   = roc_auc_score(y_add, y_proba_loo)

print(f"in-sample AUC      = {exp2_auc:.3f}")
print(f"LOO (pooled) AUC   = {exp2_auc_loo:.3f}")
print(f"slope on semantic_similarity (scaled) = {exp2_results.coef_[0,0]:+.3f}  "
      f"(negative => high |sem_sim| reduces P(additive) — the geometric prediction)")

in-sample AUC      = 0.646
LOO (pooled) AUC   = 0.253
slope on semantic_similarity (scaled) = -0.725  (negative => high |sem_sim| reduces P(additive) — the geometric prediction)


In [87]:
# Run CI bootstrap and p-value permutation
stats = bootstrap_perm(binary_model, X_sem_s, y_add, exp2_auc)
results.append({
    "experiment":  "2_sem_sim",
    "outcome":     "is_additive",
    "predictors":  "semantic_similarity",
    "n_features":  1,
    "auc_loo":     exp2_auc_loo,
    "slope_sem_sim": float(exp2_results.coef_[0, 0]),
    "slope_both_antisocial": None,         # not in this model
    "slope_signed_cos":      None,
    **stats,
})

AUC = 0.646  95% CI [0.444, 0.939]  (n_boot=2000)
permutation p (one-sided) = 0.4413
